In [1]:
# Run garbage collection
import gc
gc.collect()

0

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import wfdb
import ast
import torch
import seaborn as sns
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from tabulate import tabulate
import os
from collections import Counter
from sklearn.preprocessing import StandardScaler

In [3]:
import scipy.signal as signal
def filter_signal(signal_data, axis=0, fs=100, lowcut=0.5, highcut=45.0, notch_freq=50.0, Q=30.0):
        nyquist = 0.5 * fs
        low = lowcut / nyquist
        high = highcut / nyquist
        b, a = signal.butter(4, [low, high], btype='band')
        notch_b, notch_a = signal.iirnotch(notch_freq / nyquist, Q)
        filtered_signal = signal.filtfilt(b, a, signal_data, axis=axis)
        filtered_signal = signal.filtfilt(notch_b, notch_a, filtered_signal, axis=axis)
        return filtered_signal

In [4]:
def normalize_data_z_score(data: np.ndarray) -> np.ndarray:
    num_samples, sample_length, num_channels = data.shape
    data_reshaped = data.reshape(-1, num_channels)
    scaler = StandardScaler()
    data_normalized = scaler.fit_transform(data_reshaped)
    return data_normalized.reshape(num_samples, sample_length, num_channels)

In [5]:
path_load = './data_source_sub'


##train
X_train = np.load(os.path.join(path_load, 'Y_train.npy'))
X_train = normalize_data_z_score(X_train)
#X_train = filter_signal(X_train, axis=1, fs=100, lowcut=0.5, highcut=45.0)

y_train = pd.read_csv(os.path.join(path_load, 'Z_train.csv'))
y_train = y_train.drop(columns=['ecg_id'])
y_train = y_train.to_numpy(dtype=np.float32)

z_train = pd.read_csv(os.path.join(path_load, 'T_train.csv'))
z_train = z_train.drop(columns=['ecg_id'])
z_train = z_train.to_numpy(dtype=np.float32)

##test
X_test = np.load(os.path.join(path_load, 'Y_test.npy'))
X_test = normalize_data_z_score(X_test)
#X_test = filter_signal(X_test, axis=1, fs=100, lowcut=0.5, highcut=45.0)

y_test = pd.read_csv(os.path.join(path_load, 'Z_test.csv'))
y_test = y_test.drop(columns=['ecg_id'])
y_test = y_test.to_numpy(dtype=np.float32)

z_test = pd.read_csv(os.path.join(path_load, 'T_test.csv'))
z_test = z_test.drop(columns=['ecg_id'])
z_test = z_test.to_numpy(dtype=np.float32)

##validation
X_val = np.load(os.path.join(path_load, 'Y_valid.npy'))
X_val = normalize_data_z_score(X_val)
#X_val = filter_signal(X_val, axis=1, fs=100, lowcut=0.5, highcut=45.0)

y_val = pd.read_csv(os.path.join(path_load, 'Z_valid.csv'))
y_val = y_val.drop(columns=['ecg_id'])
y_val = y_val.to_numpy(dtype=np.float32)

z_val = pd.read_csv(os.path.join(path_load, 'T_valid.csv'))
z_val = z_val.drop(columns=['ecg_id'])
z_val = z_val.to_numpy(dtype=np.float32)

table = [
    ["X_train", X_train.shape],
    ["y_train", y_train.shape],
    ["z_train", z_train.shape],
    ["X_test", X_test.shape],
    ["y_test", y_test.shape],
    ["z_test", z_test.shape],
    ["X_val", X_val.shape],
    ["y_val", y_val.shape],
    ["z_val", z_val.shape],
]

print(tabulate(table, headers=["Dataset", "Shape"], tablefmt="grid"))

+-----------+-------------------+
| Dataset   | Shape             |
+===========+===================+
| X_train   | (17418, 1000, 12) |
+-----------+-------------------+
| y_train   | (17418, 5)        |
+-----------+-------------------+
| z_train   | (17418, 23)       |
+-----------+-------------------+
| X_test    | (2198, 1000, 12)  |
+-----------+-------------------+
| y_test    | (2198, 5)         |
+-----------+-------------------+
| z_test    | (2198, 23)        |
+-----------+-------------------+
| X_val     | (2183, 1000, 12)  |
+-----------+-------------------+
| y_val     | (2183, 5)         |
+-----------+-------------------+
| z_val     | (2183, 23)        |
+-----------+-------------------+


In [6]:
def detect_r_peaks(ecg_, fs):
    """
    Simple R-peak detector using diff, squaring, moving window integration, and adaptive thresholding.
    Returns: indices of detected R-peaks.
    """
    def simple_diff(signal):
        diff = []
        for i in range(1, len(signal)):
            diff.append(signal[i] - signal[i-1])
        return np.array(diff)

    diff = simple_diff(ecg_)
    diff = np.append(diff, 0)  # Append zero to match original length

    # Step 2: Squaring
    squared = diff ** 2

    # Step 3: Moving window integration
    window_size = int(0.150 * fs)  # 150 ms window
    integrated = np.convolve(squared, np.ones(window_size) / window_size, mode='same')

    # Step 4: Simple adaptive thresholding and refractory period
    threshold = np.mean(integrated) * 1.5
    refractory_period = int(0.10 * fs)  # 250 ms

    peaks = []
    last_peak = -refractory_period

    for i in range(1, len(integrated) - 1):
        if integrated[i] > threshold and integrated[i] > integrated[i - 1] and integrated[i] > integrated[i + 1]:
            if i - last_peak > refractory_period:
                # Local search in raw signal for actual peak in a small window
                window = ecg_[i-10:i+10]
                if len(window) == 20:
                    true_peak = i - 10 + np.argmax(window)
                    peaks.append(true_peak)
                    last_peak = true_peak
    return np.array(peaks), integrated, squared, diff

def segment_PQRST(ecg, r_peaks, start_idx=40, end_idx=100):
    # The shape of ecg is (1, 1000, 12), so length is 1000 and number of channels is 12
    length = 1000
    PQRSTs = []
    expected_segment_length = start_idx + end_idx
    plt.figure()
    # Iterate through each lead (channel) first
    for lead_idx in range(ecg.shape[2]):  # Loop through each lead (channel)
        segments_for_this_lead = []
        #plt.figure()
        # Iterate through each R-peak and extract segments for this lead
        for i, r_peak in enumerate(r_peaks):
            start_beat = np.max([0, r_peak - start_idx])
            end_beat = np.min([r_peak + end_idx, length])
            if end_beat == length:
                continue
            # Check if we need padding at the start of the segment (if r_peak - start_idx is negative)
            if r_peak - start_idx < 0:
                start_padding = np.zeros(abs(r_peak - start_idx))  # Create padding at the start
                segment = ecg[0, start_beat:end_beat, lead_idx]  # Extract the segment
                segment = np.concatenate((start_padding, segment))  # Add start padding
            else:
                # Extract the segment without start padding if no negative index
                segment = ecg[0, start_beat:end_beat, lead_idx]
            
            # Pad the segment if it's shorter than the expected segment length at the end
            if expected_segment_length > len(segment):
                padding = np.zeros(expected_segment_length - len(segment))  # Create padding
                segment = np.concatenate((segment, padding))  # Pad the segment

            # Make sure the segment has the expected length
            assert len(segment) == expected_segment_length, f"Segment length mismatch: {len(segment)} != {expected_segment_length}"

            segments_for_this_lead.append(segment)
            
            # Plotting the segment for this lead (optional)
            plt.plot(segment)
        # plt.title(f"Lead {lead_idx+1}, R-peak {i+1}")
        # plt.xlabel('Time')
        # plt.ylabel('Amplitude')
        # plt.grid()
        
        # Add the segments for this lead (as a 2D array)
        PQRSTs.append(np.array(segments_for_this_lead))
    plt.show()
    return PQRSTs

def compute_mean_std(PQRSTs):
    # Convert PQRSTs into a NumPy array (12 leads, 4 R-peaks, 250 samples per segment)
    PQRSTs = np.array(PQRSTs)  # Shape: (12, 4, 250)
    
    # Compute mean and std for each lead across R-peaks (axis 1) while preserving the time axis (axis 2)
    means = np.mean(PQRSTs, axis=1, keepdims=True)  # Mean across R-peaks (12, 1, 250)
    stds = np.std(PQRSTs, axis=1, keepdims=True)    # Std across R-peaks (12, 1, 250)
    
    return means, stds

In [7]:
def extract_mean_stds(dataset, fs = 100):
    all_means = []
    all_stds = []
    for i in range(dataset.shape[0]):
        # Select a random sample (replace 'ran' with a valid index or variable)
        ecg = dataset[i, :, :]
        ecg = np.expand_dims(ecg, axis = 0)
        # Assuming you're extracting Lead II for R-peaks detection
        ecg_leadII = ecg[0, :, 1]
    
        # Detect R-peaks
        r_peaks, _, _, _ = detect_r_peaks(ecg_leadII, fs)
    
        # Segment the PQRST segments
        PQRST_segment = segment_PQRST(ecg, r_peaks)
    
        # Compute mean and std for the segmented PQRSTs
        means, stds = compute_mean_std(PQRST_segment)
    
        if means.shape == (12, 1, 250):
            # Reshape if necessary, but keep them in the correct shape (1, 250, 12)
            means = means.reshape(1, 250, 12)  # Reshape means to (1, 250, 12)
            stds = stds.reshape(1, 250, 12)    # Reshape stds to (1, 250, 12)
        else:
            print(f"Index {i} has no means, stds")
            means = np.zeros((1, 250, 12))
            stds = np.zeros((1, 250, 12))  
        # Append the results for each sample
        all_means.append(means)
        all_stds.append(stds)

    # Convert lists to arrays at the end
    all_means = np.concatenate(all_means, axis=0)  # Shape will be (B, 250, 12) where B is the number of samples
    all_stds = np.concatenate(all_stds, axis=0)    # Shape will be (B, 250, 12)

    # Check the shape of the final arrays
    print("Means shape:", all_means.shape)  # Expected shape: (B, 250, 12)
    print("Stds shape:", all_stds.shape)    # Expected shape: (B, 250, 12)
    return all_means, all_stds

In [ ]:
import os
means, stds = extract_mean_stds(X_val,fs=100)

In [17]:
lead_indices = [0, 1, 6, 7, 8, 9, 10, 11]  # Assuming I=0, II=1, V1=6,...,V6=11
ecg = X_test[:, :, lead_indices]

In [269]:

# Example label_map and label_map_sub
label_map = {
    'NORM': 0, 'CD': 1, 'HYP': 2, 'MI': 3, 'STTC': 4
}

label_map_sub = {
    'NORM': 0, 'LAFB/LPFB': 1, 'IRBBB': 2, 'ILBBB': 3, 'CLBBB': 4,
    'CRBBB': 5, '_AVB': 6, 'WPW': 7, 'LVH': 8, 'LAO/LAE': 9, 
    'RAO/RAE': 10, 'AMI': 11, 'IMI': 12, 'LMI': 13, 'ISCA': 14, 
    'ISCI': 15, 'ISC_': 16, 'STTC': 17, 'NST_': 18
}


In [286]:
classes = 0

In [1]:
indices = np.where(y_test[:, classes] == 1.0)[0]
index = np.random.choice(indices)
fs = 100
ecg_choose = ecg[index, :, :]
ecg_leadII = ecg_choose[:, 1]
r_peaks, _, _, _ = detect_r_peaks(ecg_leadII, fs)
ecg_choose = np.expand_dims(ecg_choose, axis=0)
PQRST_segment = segment_PQRST(ecg_choose, r_peaks)

PQRST_segment_numpy = np.array(PQRST_segment)

PQRST_segment_numpy = PQRST_segment_numpy.transpose(1, 0, 2)
dower_matrix = np.array([
    [0.156, 0.037, -0.172],   # I
    [0.189, -0.310, -0.246],  # II
    [-0.097, 0.182, 0.231],   # V1
    [-0.072, 0.320, 0.136],   # V2
    [-0.189, 0.240, -0.141],  # V3
    [-0.114, 0.155, -0.073],  # V4
    [-0.022, 0.052, -0.018],  # V5
    [0.041, -0.062, -0.022],  # V6
])  # Shape: (8 leads, 3 VCG axes)


vcg = np.einsum('...lt,lc->...ct', PQRST_segment_numpy, dower_matrix)  # Result: (3, T)

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Initialize the dictionary to store results
results = []
fig = plt.figure(figsize=(24,12))
ax = fig.add_subplot(211, projection='3d')
ax2 = fig.add_subplot(221)
# Loop through each VCG instance (QRS complex)
for i in range(vcg.shape[0]):
    # Transpose and fit PCA
    vcg_transpose = vcg.transpose(0, 2, 1)[i]
    pca = PCA(n_components=3)
    pca.fit(vcg_transpose)
    explained_var = pca.explained_variance_ratio_  # array of shape (3,)
    components = pca.components_  # shape: (3, 3), each row = principal axis
    lam1, lam2, lam3 = pca.explained_variance_
    
    # Calculate Planarity, Nonplanarity, and Area
    planarity = lam2 / lam1
    nonplanarity = lam3 / lam1
    area = np.sqrt(lam1 * lam2)
    
    # Calculate QRS–T spatial angle
    qrs_start = 35
    qrs_end = 45
    t_start = 60
    t_end = 80
    st_j_offset = 45
    
    qrs_vec = vcg[i, :, qrs_start:qrs_end].mean(axis=1)
    t_vec = vcg[i, :, t_start:t_end].mean(axis=1)
    cos_theta = np.dot(qrs_vec, t_vec) / (np.linalg.norm(qrs_vec) * np.linalg.norm(t_vec))
    cos_theta = np.clip(cos_theta, -1.0, 1.0)  # for numerical safety
    qrs_t_angle_deg = np.degrees(np.arccos(cos_theta))
    
    # QRS loop planarity via PCA
    qrs_loop = vcg[i, :, qrs_start:qrs_end].T  # shape (N, 3)
    pca = PCA(n_components=3).fit(qrs_loop)
    qrs_planarity = pca.explained_variance_[1] / pca.explained_variance_[0]
    
    # QRS duration (in ms)
    sampling_rate = 500  # Hz
    qrs_duration_ms = (qrs_end - qrs_start) * 1000 / sampling_rate
    
    # ST vector (J-point + 10 ms)
    st_vector = vcg[i, :, st_j_offset]
    
    # T-loop polarity (dot product sign between T and QRS vector)
    t_loop = vcg[i, :, t_start:t_end]
    t_pca = PCA(n_components=1).fit(t_loop.T)
    t_axis = t_pca.components_[0]
    t_polarity = np.sign(np.dot(t_axis, qrs_vec))
    
    # Store results in the dictionary
    result = {
        'QRS-T Spatial Angle (deg)': round(qrs_t_angle_deg, 2),
        'QRS Loop Planarity': round(qrs_planarity, 4),
        #'QRS Duration (ms)': qrs_duration_ms,
        'ST Vector': np.round(st_vector, 3),
        'T-loop Polarity (relative to QRS)': int(t_polarity),
        'Explained Variance Ratio': explained_var.tolist(),
        'Area': area
    }
    
    # Append result for this QRS
    results.append(result)

    # Optional: Plot 3D VCG loop
    ax.plot(vcg_transpose[:, 0], vcg_transpose[:, 1], vcg_transpose[:, 2], label=f'VCG Loop {i}')
    vcg_magnitude = np.sqrt(vcg_transpose[:, 0]**2 + vcg_transpose[:, 1]**2 + vcg_transpose[:, 2]**2)
    ax2.plot(vcg_magnitude)
    
ax.set_title("QRS Loop in 3D (VCG)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.legend()
ax2.grid(True)
plt.show()
# Step 2: Convert results into a Pandas DataFrame
df = pd.DataFrame(results)
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Disable line wrapping
pd.set_option('display.max_colwidth', None)  # Display full column content
super_class_indices = np.where(y_test[index] == 1)[0]
super_class_labels = [label for label, idx in label_map.items() if idx in super_class_indices]

# Extract sub class labels (all indices where 1's appear)
sub_class_indices = np.where(z_test[index] == 1)[0]
sub_class_labels = [label for label, idx in label_map_sub.items() if idx in sub_class_indices]


# Print the DataFrame in a tabular manner
print(f"Supper Class: {super_class_labels}, Sub Class: {sub_class_labels}")
print(df)


NameError: name 'np' is not defined